# Cats vs. Dogs Classification with a Custom CNN from Scratch

Welcome to deep learning! In this notebook, we build, train, and test our own **Convolutional Neural Network (CNN)** from the ground up using **PyTorch** and the official **Kaggle Dogs vs. Cats** dataset.

### What we will do:
1. **Explore the Kaggle Dataset**: 20,000 training photos + 5,000 validation photos.
2. **Data Pipeline (DataLoaders & Transforms)**: Resize to 128x128, augment (flips/rotations), and normalize.
3. **Build the CNN Architecture from Scratch**: Custom multi-layer neural network (`Conv2d`, `ReLU`, `MaxPool2d`, `Linear`).
4. **Train the Neural Network**: Forward pass -> Compute Loss -> Backpropagation -> Update Weights.
5. **Test on Any Photo**: Feed an image of a cat or dog and see the model predict the label and confidence percentage!

In [ ]:
# Step 1: Import Essential Libraries
import os
import time
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# Device configuration (MPS for Apple Silicon Mac, CUDA for Nvidia GPU, or CPU)
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using compute device: {device}")

## Step 2: Verify the Kaggle Dataset
We check the organized Kaggle dataset folders in `dataset/cats_and_dogs/`.

In [ ]:
# Setup directories
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TARGET_DIR = PROJECT_ROOT / "dataset" / "cats_and_dogs"

if not TARGET_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found at: {TARGET_DIR}")

train_cats = len(list((TARGET_DIR / "train" / "cats").glob("*.jpg")))
train_dogs = len(list((TARGET_DIR / "train" / "dogs").glob("*.jpg")))
val_cats = len(list((TARGET_DIR / "validation" / "cats").glob("*.jpg")))
val_dogs = len(list((TARGET_DIR / "validation" / "dogs").glob("*.jpg")))

print("=" * 45)
print(f"Dataset Path: {TARGET_DIR}")
print(f"Training Set:   {train_cats} Cats | {train_dogs} Dogs (Total: {train_cats + train_dogs})")
print(f"Validation Set: {val_cats} Cats | {val_dogs} Dogs (Total: {val_cats + val_dogs})")
print("=" * 45)

## Step 3: Inspect Sample Images
Let's visualize sample training images from both classes.

In [ ]:
cat_images = list((TARGET_DIR / "train" / "cats").glob("*.jpg"))[:4]
dog_images = list((TARGET_DIR / "train" / "dogs").glob("*.jpg"))[:4]

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i, cat_path in enumerate(cat_images):
    img = Image.open(cat_path)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"Cat: {img.size}")
    axes[0, i].axis("off")

for i, dog_path in enumerate(dog_images):
    img = Image.open(dog_path)
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"Dog: {img.size}")
    axes[1, i].axis("off")

plt.suptitle("Sample Kaggle Training Images (Cats vs. Dogs)", fontsize=14)
plt.tight_layout()
plt.show()

## Step 4: Data Preprocessing & PyTorch DataLoaders
Raw photos have different dimensions (e.g., 300x400, 500x375). We standardize and transform them:
1. **`Resize((128, 128))`**: Standardizes dimensions to 128x128.
2. **`RandomHorizontalFlip()`**: Data augmentation (a flipped cat is still a cat).
3. **`ToTensor()`**: Scales pixels from [0, 255] to [0.0, 1.0].
4. **`Normalize()`**: Centers pixel values for stable mathematical convergence.

> **Tip**: You can set `SUBSET_SIZE = 2000` to do a quick 2-minute test run, or `None` to train on the full 20,000 images!

In [ ]:
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32

# Optional: Set to 2000 for fast testing in 2 mins, or None for full 20,000 dataset
SUBSET_SIZE = 2000

# Training transformations (with data augmentation)
train_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation transformations (no random augmentations)
val_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load full datasets using PyTorch ImageFolder
full_train_dataset = datasets.ImageFolder(str(TARGET_DIR / "train"), transform=train_transforms)
full_val_dataset = datasets.ImageFolder(str(TARGET_DIR / "validation"), transform=val_transforms)

class_names = full_train_dataset.classes
print("Class labels mapped:", full_train_dataset.class_to_idx)

# Select subset or full dataset
if SUBSET_SIZE is not None and SUBSET_SIZE < len(full_train_dataset):
    train_indices = random.sample(range(len(full_train_dataset)), SUBSET_SIZE)
    val_indices = random.sample(range(len(full_val_dataset)), SUBSET_SIZE // 4)
    train_dataset = Subset(full_train_dataset, train_indices)
    val_dataset = Subset(full_val_dataset, val_indices)
    print(f"Using fast subset: {len(train_dataset)} train images, {len(val_dataset)} val images.")
else:
    train_dataset = full_train_dataset
    val_dataset = full_val_dataset
    print(f"Using full Kaggle dataset: {len(train_dataset)} train images, {len(val_dataset)} val images.")

# DataLoaders (batching images)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)} | Validation batches: {len(val_loader)}")

## Step 5: Building the CNN Model from Scratch
Here we design our custom Convolutional Neural Network architecture!

### Architecture:
* **Input**: `[3 channels, 128 height, 128 width]`
* **Block 1**: `Conv2d(3 -> 32)` -> `BatchNorm` -> `ReLU` -> `MaxPool2d(2)` (Size: 64x64)
* **Block 2**: `Conv2d(32 -> 64)` -> `BatchNorm` -> `ReLU` -> `MaxPool2d(2)` (Size: 32x32)
* **Block 3**: `Conv2d(64 -> 128)` -> `BatchNorm` -> `ReLU` -> `MaxPool2d(2)` (Size: 16x16)
* **Classifier**: Flatten -> `Linear(128 * 16 * 16 -> 128)` -> `Dropout(0.5)` -> `Linear(128 -> 2)`

In [ ]:
class CatDogCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(CatDogCNN, self).__init__()
        
        # 1. Feature Extractor (Convolutional layers)
        self.features = nn.Sequential(
            # Block 1: 128x128 -> 64x64
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 2: 64x64 -> 32x32
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 3: 32x32 -> 16x16
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        # 2. Classifier (Dense Decision Maker)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5), # Regularization: prevents overfitting
            nn.Linear(128, num_classes) # Outputs 2 numbers: [score_cat, score_dog]
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create model instance and move to compute device
model = CatDogCNN(num_classes=2).to(device)
print("Custom CNN Model Architecture:")
print(model)

## Step 6: Loss Function & Optimizer
* **`nn.CrossEntropyLoss`**: Computes the difference between predicted probabilities and actual labels.
* **`optim.Adam`**: Dynamically adjusts internal weights using gradient descent.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Criterion:", criterion)
print("Optimizer:", optimizer)

## Step 7: Training & Validation Loop
In each **Epoch**:
1. **Training Phase**: Feed batch -> Forward Pass -> Compute Loss -> Backpropagation (`loss.backward()`) -> Update weights (`optimizer.step()`).
2. **Validation Phase**: Evaluate on unseen photos without updating weights, and calculate accuracy.

In [ ]:
NUM_EPOCHS = 6
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print(f"Starting training for {NUM_EPOCHS} epochs on {device}...")
start_total = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    
    # --- 1. Training Phase ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()            # Reset gradients
        outputs = model(images)           # Forward pass (guess)
        loss = criterion(outputs, labels) # Calculate error
        loss.backward()                  # Backward pass (calculate gradients)
        optimizer.step()                 # Nudge weights
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)
        
    train_loss = running_loss / total_train
    train_acc = (correct_train / total_train) * 100
    
    # --- 2. Validation Phase ---
    model.eval()
    val_loss_running = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad(): # Disable gradients for evaluation speed
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss_running += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct_val += (predicted == labels).sum().item()
            total_val += labels.size(0)
            
    val_loss = val_loss_running / total_val
    val_acc = (correct_val / total_val) * 100
    
    # Record metrics
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    epoch_time = time.time() - epoch_start
    print(f"Epoch [{epoch:2d}/{NUM_EPOCHS:2d}] ({epoch_time:.1f}s) | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

print(f"\nTraining completed in {(time.time() - start_total)/60:.2f} minutes!")

## Step 8: Plot Learning Curves
Let's visualize the loss reduction and accuracy improvement over epochs.

In [ ]:
plt.figure(figsize=(12, 4))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(range(1, NUM_EPOCHS + 1), history["train_loss"], label="Train Loss", marker="o")
plt.plot(range(1, NUM_EPOCHS + 1), history["val_loss"], label="Val Loss", marker="o")
plt.title("Training vs. Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(range(1, NUM_EPOCHS + 1), history["train_acc"], label="Train Accuracy (%)", marker="o", color="green")
plt.plot(range(1, NUM_EPOCHS + 1), history["val_acc"], label="Val Accuracy (%)", marker="o", color="orange")
plt.title("Training vs. Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Step 9: Testing on Unseen Photos (Inference)
We test our trained model on individual test photos and print the predicted probability!

In [ ]:
def predict_image(image_path, model, class_names):
    """Loads an image, runs inference, and displays prediction with confidence %."""
    img_raw = Image.open(image_path).convert("RGB")
    tensor = val_transforms(img_raw).unsqueeze(0).to(device) # Shape: [1, 3, 128, 128]
    
    model.eval()
    with torch.no_grad():
        outputs = model(tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        pred_idx = torch.argmax(probabilities).item()
        confidence = probabilities[pred_idx].item() * 100
        
    predicted_label = class_names[pred_idx]
    
    # Plot
    plt.figure(figsize=(4, 4))
    plt.imshow(img_raw)
    color = "green" if "cat" in predicted_label.lower() else "blue"
    plt.title(f"Prediction: {predicted_label.upper()} ({confidence:.1f}%)", color=color, fontsize=12, fontweight="bold")
    plt.axis("off")
    plt.show()
    
    print(f"Probabilities -> Cat: {probabilities[0]*100:.2f}% | Dog: {probabilities[1]*100:.2f}%")

# Test on sample photos from the validation folder
test_cat = list((TARGET_DIR / "validation" / "cats").glob("*.jpg"))[0]
test_dog = list((TARGET_DIR / "validation" / "dogs").glob("*.jpg"))[0]

print("--- Testing Sample Cat ---")
predict_image(test_cat, model, class_names)

print("--- Testing Sample Dog ---")
predict_image(test_dog, model, class_names)

## Step 10: Save Trained Weights
Export model weights so you can reload your trained neural network anytime!

In [ ]:
MODEL_SAVE_DIR = PROJECT_ROOT / "runs" / "cat_dog_model"
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

SAVE_PATH = MODEL_SAVE_DIR / "cat_dog_cnn_scratch.pth"
torch.save(model.state_dict(), str(SAVE_PATH))
print(f"Trained model weights saved successfully to: {SAVE_PATH}")